# Cohort pipeline · all nights

Until now we worked on **one night**. This notebook runs the same steps on **every night you have**, and collects the results in **one table, one row per night**: the table we use for the age analyses and your dashboard.

This is the **complete version**: it has all the blocks of the week.

Each session added a block of features to `process_night`:
- **Session 4:** sleep statistics (on the unedited hypnogram)
- **Session 5:** band power per stage (Section 2.1)
- **Session 6:** automated staging with YASA vs. the experts, with entropy, instability (KL divergence) and intrusions (Section 2.3)
- **Session 7:** spindles and slow waves (Section 2.2)

**How to work:** run from top to bottom. Cells marked 👉 have a parameter to change and rerun. Running all 153 nights takes about 5–15 min.

In [ ]:
%matplotlib inline
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa
from scipy.integrate import simpson
from scipy.signal import welch
from scipy.stats import spearmanr

sys.path.insert(0, str(Path("..").resolve()))  # so Python finds sleep_utils.py and cohort.py in the repo folder
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sleep_utils import SUBJECTS_CSV, load_continuous, load_epochs, load_subject, night_code
from cohort import sleep_architecture_features

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)  # empty means of nights without a stage

RESULTS_DIR = Path("..") / "results"
# a separate file, so the table of the first version (cohort_features.csv) stays as it is
COHORT_FILE = RESULTS_DIR / "cohort_features_final.csv"

## 1. The nights, and which ones you include

All your choices about which nights to use live in **`database.csv`** in the repo folder: one row per night (the columns of `SC-subjects.csv`, plus the night `code`), with two extra columns:
- **`include`**: `True` or `False`. The pipeline only runs the nights with `True`, and the analyses only use those.
- **`exclude_reason`**: why you left the night out. Always write one: you'll have to defend it in your presentation.

The first run creates `database.csv` with all nights included. After that, it is **never overwritten**: this notebook will get new sections during the week, and your choices stay in your file. It's in `.gitignore`, so `git pull` doesn't touch it either.

👉 To leave out a night, open `database.csv` in VS Code, set `include` to `False` and write the reason, save, and rerun from here. To leave out a person, do it for all their nights. You don't need to rerun the pipeline (Section 3): Section 4 only uses the included nights. (If you edit it in Excel, save it as CSV again.)

In [ ]:
DATABASE_FILE = Path("..") / "database.csv"
if not DATABASE_FILE.exists():  # first run: SC-subjects.csv + code, include and exclude_reason, all nights included
    subjects = pd.read_csv(SUBJECTS_CSV)
    subjects.insert(0, "code", [night_code(s, n) for s, n in zip(subjects["subject"], subjects["night"])])
    subjects["include"] = True
    subjects["exclude_reason"] = ""
    subjects.to_csv(DATABASE_FILE, index=False)
    print(f"Created {DATABASE_FILE.resolve()}")

subjects = pd.read_csv(DATABASE_FILE, dtype={"exclude_reason": str})
subjects["exclude_reason"] = subjects["exclude_reason"].fillna("")

# include must be True or False (also accepts true/false/1/0, e.g. after editing in Excel)
include = subjects["include"].astype(str).str.strip().str.lower().map({"true": True, "1": True, "false": False, "0": False})
if include.isna().any():
    raise ValueError(f"include must be True or False. Check these nights in {DATABASE_FILE.name}: "
                     f"{list(subjects.loc[include.isna(), 'code'])}")
subjects["include"] = include.astype(bool)

no_reason = subjects.loc[~subjects["include"] & (subjects["exclude_reason"].str.strip() == ""), "code"]
if len(no_reason):
    print(f"⚠️ Excluded without a reason: {list(no_reason)}")

nights = subjects[subjects["include"]].reset_index(drop=True)
print(f"{len(nights)} of {len(subjects)} nights included ({nights['subject'].nunique()} people)")
subjects.loc[~subjects["include"], ["code", "age", "exclude_reason"]]


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(subjects["age"], bins=range(20, 105, 5), color="lightgray", label="all nights")
ax.hist(nights["age"], bins=range(20, 105, 5), color="tab:blue", label="included")
ax.set(xlabel="Age (years)", ylabel="Nights")
ax.legend()
plt.show()

## 2. One function per night

`process_night` does, for one night, what you did step by step in Sessions 2–4, and returns a **dict of features** (one row of the cohort table).

- **Loading and artefacts:** `load_epochs` from `sleep_utils.py`: `load_subject` (lights-off → end of the sleep period: naps the next day are left out, see Session 4), filter 0.1–40 Hz, 30-s epochs, and the bad epochs (the course detector + movement time, `bad_epochs_table`, Session 3). The table of bad epochs is saved in `results/<night>/bad_epochs.csv` (not `bad_epochs_auto.csv`, so this morning's file with all methods stays)
- **Sleep statistics:** `sleep_architecture_features` from `cohort.py`: YASA on the **unedited** hypnogram (Session 4), plus the REM latency from sleep onset, the stage shifts per hour, the **sleep continuity index** and the **transition rate** out of each stage (Session 4, Exercises 4.2 and 5.2). Open `cohort.py` to see the code
- **Data quality:** % bad epochs, overall and per stage, and how many clean epochs are left per stage (these are what the EEG features of the next sessions are computed on)
- **Band power:** `power_features` below. Welch spectrum of every clean 30-s epoch (2-s windows), area under the PSD per band (Session 5), absolute (µV²) and relative to the total power 0.5–40 Hz. **Median over the clean epochs of each stage**, for both channels: columns `<band>_<abs|rel>_<stage>_<Fpz|Pz>`, e.g. `delta_rel_N3_Fpz`
- **Spindles and slow waves:** `microstructure_features` below. YASA on the continuous signal, in the clean N2 and N3 epochs only (Session 7): spindles in the 12–16 Hz band, slow waves, everything else YASA's defaults. Densities, amplitude, frequency, and the slow-wave–spindle coupling. The events of every night are saved in `results/<night>/spindles.csv` and `slow_waves.csv`
- **Automated staging:** `staging_features` below. YASA stages the night (Fpz-Cz + horizontal EOG, no EMG, no metadata): agreement with the experts (accuracy, kappa, F1 per stage), the sleep statistics of YASA's hypnogram (`yasa_TST`, ...), and, from its **hypnodensity** (the probability of each stage per epoch), the **entropy**, the **instability** (KL divergence between consecutive epochs) and the **intrusions** (Herzog et al., 2025). The staging of every night is saved in `results/<night>/staging_yasa.csv`

Read the functions once before you run them.

### 2.1 Band power per stage

Same definitions as Session 5: Welch with 2-s windows, bands delta 0.5–4, theta 4–8, alpha 8–12, sigma 12–16, beta 16–30, gamma 30–40 Hz, power = area under the PSD (Simpson), relative power = band / total power 0.5–40 Hz. We take the **median of the epochs** of each stage (as in Session 5), only over clean epochs.

In [ ]:
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12), "sigma": (12, 16), "beta": (16, 30), "gamma": (30, 40)}
TOTAL = (0.5, 40)
POWER_CHANNELS = {"Fpz-Cz": "Fpz", "Pz-Oz": "Pz"}  # channel -> short name in the column names
POWER_STAGES = ["WAKE", "N1", "N2", "N3", "REM"]


def band_power(freqs, psd, band):
    """Absolute power (µV²) in `band` = (low, high) Hz: the area under the PSD, along the last axis."""
    low, high = band
    in_band = (freqs >= low) & (freqs <= high)
    return simpson(psd[..., in_band], x=freqs[in_band], axis=-1)


def power_features(epochs, stages, bad):
    """Band power per stage (median of the clean epochs), as a dict: <band>_<abs|rel>_<stage>_<Fpz|Pz>."""
    sf = epochs.info["sfreq"]
    eeg = epochs.get_data(picks=list(POWER_CHANNELS), units="uV")  # (n_epochs, 2 channels, n_samples)
    freqs, psd = welch(eeg, sf, nperseg=int(2 * sf))  # 2-s windows, 50 % overlap
    keep = freqs <= 40
    freqs, psd = freqs[keep], psd[..., keep]

    total = band_power(freqs, psd, TOTAL)  # (n_epochs, 2)
    absolute = {name: band_power(freqs, psd, band) for name, band in BANDS.items()}
    clean = ~bad["bad"].to_numpy()
    features = {}
    for stage in POWER_STAGES:
        in_stage = (np.asarray(stages) == stage) & clean  # the clean epochs of this stage
        for name in BANDS:
            for k, short in enumerate(POWER_CHANNELS.values()):
                a = absolute[name][in_stage, k]
                features[f"{name}_abs_{stage}_{short}"] = np.median(a) if in_stage.any() else np.nan
                features[f"{name}_rel_{stage}_{short}"] = np.median(a / total[in_stage, k]) if in_stage.any() else np.nan
    return features

### 2.2 Spindles and slow waves

From Session 7, the course settings: spindle band **12–16 Hz** (YASA's default is 12–15 Hz), everything else YASA's defaults, in the clean N2 and N3 epochs (`load_continuous` puts −1 in the bad epochs, so YASA doesn't look there). Density = events per minute of clean N2 or N3.

In [ ]:
MICRO_CHANNELS = {"Fpz-Cz": "Fpz", "Pz-Oz": "Pz"}  # channel -> short name in the column names
MICRO_STAGES = {"N2": 2, "N3": 3}  # stage -> YASA's code


def circular_mean(phases):
    """Mean direction (radians) and its strength (0 = phases all over the place, 1 = all the same)."""
    vector = np.mean(np.exp(1j * np.asarray(phases)))  # each phase as a unit vector; their mean
    return np.angle(vector), np.abs(vector)


def microstructure_features(subject, night):
    """Spindles (12-16 Hz) and slow waves (YASA defaults) in the clean N2 and N3 epochs, as a dict.

    Per stage and channel: sp_density_<stage>_<ch>, sw_density_<stage>_<ch> (events per minute of clean sleep).
    Per channel: spindle amplitude, frequency, duration (N2); slow-wave PTP, slope, frequency (N2 + N3);
    slow-wave-spindle coupling (N2 + N3): mean phase (degrees), its strength and ndPAC.
    Saves the events of the night in results/<night>/spindles.csv and slow_waves.csv.
    """
    raw, hypno, stages, artefacts = load_continuous(subject, night, picks=list(MICRO_CHANNELS))
    sf = raw.info["sfreq"]
    sp = yasa.spindles_detect(raw, hypno=hypno, include=(2, 3), freq_sp=(12, 16))  # our band
    sw = yasa.sw_detect(raw, hypno=hypno, include=(2, 3), coupling=True)
    # YASA returns None when it finds nothing: then an empty table
    sp_events = sp.summary() if sp is not None else pd.DataFrame(columns=["Stage", "Channel"])
    sw_events = sw.summary() if sw is not None else pd.DataFrame(columns=["Stage", "Channel"])

    features = {}
    for stage, code in MICRO_STAGES.items():
        minutes = (hypno == code).sum() / sf / 60  # clean minutes of this stage
        features[f"clean_min_{stage}"] = minutes
        for ch, short in MICRO_CHANNELS.items():
            for kind, events in [("sp", sp_events), ("sw", sw_events)]:
                n = ((events["Stage"] == code) & (events["Channel"] == ch)).sum()
                features[f"{kind}_density_{stage}_{short}"] = n / minutes if minutes > 0 else np.nan

    for ch, short in MICRO_CHANNELS.items():
        s = sp_events[(sp_events["Stage"] == 2) & (sp_events["Channel"] == ch)]
        for col in ["Amplitude", "Frequency", "Duration"]:
            features[f"sp_{col.lower()}_N2_{short}"] = s[col].mean() if len(s) else np.nan
        w = sw_events[sw_events["Channel"] == ch]
        for col, name in [("PTP", "ptp"), ("Slope", "slope"), ("Frequency", "frequency")]:
            features[f"sw_{name}_{short}"] = w[col].mean() if len(w) else np.nan
        phase, strength = circular_mean(w["PhaseAtSigmaPeak"]) if len(w) else (np.nan, np.nan)
        features[f"sw_coupling_phase_{short}"] = np.degrees(phase)
        features[f"sw_coupling_strength_{short}"] = strength
        features[f"sw_ndPAC_{short}"] = w["ndPAC"].mean() if len(w) else np.nan

    folder = RESULTS_DIR / night_code(subject, night)
    folder.mkdir(parents=True, exist_ok=True)
    sp_events.to_csv(folder / "spindles.csv", index=False)
    sw_events.to_csv(folder / "slow_waves.csv", index=False)
    return features

### 2.3 Automated staging, entropy, KL divergence and intrusions

From Session 6. YASA gives, for every epoch, a probability for each stage (the **hypnodensity**). From it, per expert stage (Herzog et al., 2025):
- **Entropy** H = −Σ p·log₂ p (bits): 0 when YASA is sure, log₂ 5 ≈ 2.3 when it has no idea.
- **Instability**: the **KL divergence** D_KL(Pᵢ ‖ Pᵢ₊₁) = Σ p·log₂(p/q) between consecutive epochs of the same expert stage (bits). 0 = the next epoch has exactly the same profile.
- **Wake intrusion**: mean P(WAKE) in the expert's sleep epochs. **Sleep intrusion**: mean 1 − P(WAKE) in the expert's **wake** epochs (the expert scored wake, the EEG looks partly like sleep).

N1 is left out of the entropy and KL columns, as in Herzog et al. All the `yasa_*` columns are computed over the epochs the expert scored (ART and UNS are left out of the agreement).

In [ ]:
STAGING_STAGES = ["WAKE", "N1", "N2", "N3", "REM"]
HD_STAGES = ["WAKE", "N2", "N3", "REM"]  # the expert stages we describe (N1 left out, as in Herzog et al.)


def entropy_bits(p):
    """Shannon entropy (bits) of each row of p (n_epochs x 5 stage probabilities)."""
    p = np.clip(p, 1e-12, 1)  # log2(0) is undefined; 0 * log2(0) counts as 0
    return -(p * np.log2(p)).sum(axis=1)


def kl_bits(p, q):
    """KL divergence D_KL(p || q) in bits, row by row."""
    p, q = np.clip(p, 1e-12, 1), np.clip(q, 1e-12, 1)
    return (p * np.log2(p / q)).sum(axis=1)


def hypnodensity_features(proba, stages):
    """Entropy, instability and intrusions per expert stage, from YASA's hypnodensity (Herzog et al., 2025).

    proba: DataFrame (n_epochs x WAKE..REM), stages: the expert stage of each epoch.
    H_mean_<stage>, H_sd_<stage>: entropy (bits) of the epochs of this expert stage
    KL_mean_<stage>, KL_sd_<stage>: KL divergence (bits) between consecutive epochs, both of this stage
    wake_intrusion: mean P(WAKE) in the expert's sleep epochs; sleep_intrusion: mean 1 - P(WAKE) in its wake epochs
    """
    p = proba[STAGING_STAGES].to_numpy()  # columns: WAKE, N1, N2, N3, REM
    stages = np.asarray(stages)
    entropy = entropy_bits(p)
    kl = kl_bits(p[:-1], p[1:])  # epoch i vs. epoch i+1
    features = {}
    for st in HD_STAGES:
        in_stage = stages == st
        pairs = (stages[:-1] == st) & (stages[1:] == st)  # both epochs of the pair in this stage
        features[f"H_mean_{st}"] = entropy[in_stage].mean() if in_stage.any() else np.nan
        features[f"H_sd_{st}"] = entropy[in_stage].std() if in_stage.any() else np.nan
        features[f"KL_mean_{st}"] = kl[pairs].mean() if pairs.any() else np.nan
        features[f"KL_sd_{st}"] = kl[pairs].std() if pairs.any() else np.nan
    sleep, wake = np.isin(stages, ["N1", "N2", "N3", "REM"]), stages == "WAKE"
    features["wake_intrusion"] = p[sleep, 0].mean()  # column 0 = WAKE
    features["sleep_intrusion"] = 1 - p[wake, 0].mean() if wake.any() else np.nan
    return features


def staging_features(subject, night, stages):
    """YASA's staging (Fpz-Cz + EOG, no EMG, no metadata) vs. the expert hypnogram `stages`, as a dict.

    yasa_kappa, yasa_accuracy, yasa_F1_<stage>, yasa_confidence (mean probability of the chosen stage), the
    sleep statistics of the YASA hypnogram (yasa_TST, yasa_%N3, ...) and the hypnodensity features
    (yasa_H_..., yasa_KL_..., yasa_wake_intrusion, yasa_sleep_intrusion).
    Saves results/<night>/staging_yasa.csv.
    """
    raw = load_subject(subject, night)  # unfiltered: YASA filters itself
    hypno = yasa.SleepStaging(raw, eeg_name="Fpz-Cz", eog_name="horizontal").predict()
    auto = hypno.hypno.to_numpy().astype(str)
    stages = np.asarray(stages)
    assert len(auto) == len(stages), "YASA and the expert hypnogram have different numbers of epochs"

    scored = np.isin(stages, STAGING_STAGES)  # the expert's ART and UNS epochs are left out
    e, p = stages[scored], auto[scored]
    features = {"yasa_accuracy": accuracy_score(e, p), "yasa_kappa": cohen_kappa_score(e, p),
                "yasa_confidence": hypno.proba.max(axis=1).mean()}
    f1 = f1_score(e, p, labels=STAGING_STAGES, average=None, zero_division=np.nan)
    features.update({f"yasa_F1_{s}": v for s, v in zip(STAGING_STAGES, f1)})
    features.update({f"yasa_{k}": v for k, v in sleep_architecture_features(auto).items()})
    features.update({f"yasa_{k}": v for k, v in hypnodensity_features(hypno.proba, stages).items()})

    table = hypno.proba.add_prefix("P_").reset_index(drop=True)
    table.insert(0, "onset", hypno.hypno.index.strftime("%H:%M:%S"))
    table.insert(1, "expert", stages)
    table.insert(2, "yasa", auto)
    folder = RESULTS_DIR / night_code(subject, night)
    folder.mkdir(parents=True, exist_ok=True)
    table.to_csv(folder / "staging_yasa.csv", index=False)
    return features

### 2.4 Put it together: `process_night`

In [ ]:
def process_night(subject, night):
    """All features of one night, as a dict."""
    code = night_code(subject, night)
    row = subjects[(subjects["subject"] == subject) & (subjects["night"] == night)].iloc[0]
    features = {"code": code, "subject": subject, "night": night, "age": row["age"],
                "sex": "F" if row["sex_F"] == 1 else "M"}

    # --- Loading + artefacts (Sessions 2-3): load, filter 0.1-40 Hz, 30-s epochs, bad epochs
    # The bad epochs are only for the EEG features, never for the sleep statistics
    epochs, stages, bad = load_epochs(subject, night)
    (RESULTS_DIR / code).mkdir(parents=True, exist_ok=True)
    bad.to_csv(RESULTS_DIR / code / "bad_epochs.csv", index=False)

    # --- Sleep statistics (Session 4): the unedited expert hypnogram
    features.update(sleep_architecture_features(stages))

    # --- Data quality
    features["pct_bad"] = 100 * bad["bad"].mean()
    for stage in ["WAKE", "N2", "N3", "REM"]:
        in_stage = bad["stage"] == stage
        features[f"pct_bad_{stage}"] = 100 * bad.loc[in_stage, "bad"].mean() if in_stage.any() else np.nan
        features[f"n_clean_{stage}"] = (in_stage & ~bad["bad"]).sum()

    # --- Band power per stage (Session 5), on the clean epochs only
    features.update(power_features(epochs, stages, bad))

    # --- Spindles and slow waves (Session 7), on the clean epochs only
    features.update(microstructure_features(subject, night))

    # --- Automated staging (Session 6): YASA vs. the experts, entropy, KL divergence, intrusions
    features.update(staging_features(subject, night, stages))

    return features

### Test it on one night first

Always test on a night you know. For SC4002 you should find the numbers of this morning (53 bad epochs, 5.2 %) and this afternoon (TST 472 min, SE 92.3 %).

The new columns for SC4002 should match the precomputed table `precomputed/cohort_microstructure.csv` (`sp_density_N2_Fpz` 1.36, `sw_density_N3_Fpz` 14.94). The `yasa_*` columns should match `precomputed/cohort_staging.csv`. One night now takes longer: it is loaded three times (spectra, staging, event detection).

In [ ]:
t0 = time.time()
test = process_night(0, 2)
print(f"{time.time() - t0:.1f} s")
pd.Series(test)

## 3. Run all nights

👉 `N_NIGHTS = None` runs all included nights (about 20–30 min now, with the spectra, the staging and the event detection). For a quick check, set `N_NIGHTS = 5` first.

If a night fails, the loop doesn't stop: it notes the error and goes on. Look at the failures at the end.

In [ ]:
N_NIGHTS = None  # 👉 None = all nights, a number = the first nights only

todo = nights if N_NIGHTS is None else nights.head(N_NIGHTS)
rows, failed = [], {}
t0 = time.time()
for k, (subject, night) in enumerate(zip(todo["subject"], todo["night"]), start=1):
    try:
        rows.append(process_night(subject, night))
    except Exception as err:  # keep going, but remember what failed
        failed[night_code(subject, night)] = repr(err)
    print(f"\r{k}/{len(todo)} nights, {time.time() - t0:.0f} s", end="")

cohort = pd.DataFrame(rows)
cohort.to_csv(COHORT_FILE, index=False)
print(f"\nSaved {len(cohort)} nights to {COHORT_FILE}. Failed: {len(failed)}")
failed

## 4. Explore the cohort

From here on, nothing is recomputed: we read the saved table and keep the included nights. After a restart, run the setup cell and Section 1, then start here.

In [ ]:
cohort = pd.read_csv(COHORT_FILE)
cohort = cohort[cohort["code"].isin(nights["code"])]  # only the nights included in Section 1
print(f"{len(cohort)} nights, {cohort['subject'].nunique()} subjects")
cohort.describe().T.round(1)

### Or: use the precomputed table

If you haven't run Section 3 (or want the instructor's numbers), read `precomputed/cohort_features.csv` instead: all 153 nights, the same columns. Everything below (plots, correlations) then uses that table. Set it back to `False` to use your own run.

In [ ]:
USE_PRECOMPUTED = False  # 👉 True: the instructor's table, instead of your own run

if USE_PRECOMPUTED:
    cohort = pd.read_csv(Path("..") / "precomputed" / "cohort_features.csv")
    cohort = cohort[cohort["code"].isin(nights["code"])]  # still only the nights included in Section 1
    print(f"Precomputed table: {len(cohort)} nights, {cohort['subject'].nunique()} subjects, {cohort.shape[1]} columns")

### One statistic vs. age

👉 Change `METRIC` and rerun: try `SE`, `TST`, `WASO`, `%N3`, `%REM`, `N3`, `SOL`, `Lat_REM_onset`, `shifts_per_h`, `SFI`, `SCI`, `rate_N1`, `rate_N3`, `rate_REM`. ρ is Spearman's rank correlation with age.

In [ ]:
METRIC = "%N3"  # 👉 change me

fig, ax = plt.subplots(figsize=(6, 4))
for sex, color in [("F", "tab:red"), ("M", "tab:blue")]:
    d = cohort[cohort["sex"] == sex]
    ax.scatter(d["age"], d[METRIC], s=18, alpha=0.7, color=color, label=sex)
rho, p = spearmanr(cohort["age"], cohort[METRIC], nan_policy="omit")
ax.set(xlabel="Age (years)", ylabel=METRIC, title=f"{METRIC} vs. age: ρ = {rho:.2f} (p = {p:.1g}, n = {len(cohort)} nights)")
ax.legend(title="Sex")
plt.show()

### All statistics at once

In [ ]:
METRICS = ["TST", "SE", "SOL", "WASO", "Lat_REM_onset", "%N1", "%N2", "%N3", "%REM", "SFI",
           "shifts_per_h", "SCI", "rate_N1", "rate_N2", "rate_REM"]

fig, axes = plt.subplots(3, 5, figsize=(16, 9), sharex=True)
for ax, m in zip(axes.flat, METRICS):
    ax.scatter(cohort["age"], cohort[m], s=8, alpha=0.6)
    rho, _ = spearmanr(cohort["age"], cohort[m], nan_policy="omit")
    ax.set_title(f"{m} (ρ = {rho:.2f})")
for ax in axes[-1]:
    ax.set_xlabel("Age (years)")
plt.tight_layout()
plt.show()

**Questions**
1. Which statistics change most with age? Does that match what you know (T2)?
2. `SCI`, the transition rates and `shifts_per_h` all describe fragmentation. Do they carry the same information? Plot two of them against each other.
3. Most subjects have **two nights**. Each dot is a night, not a person. Why is that a problem for the p-value above? (Wednesday's age prediction must handle this too.)
4. Which statistics depend on our choice of the end of the night (the sleep period ends at the first wake bout of ≥ 3 h, see `load_subject`)? Would their age trend survive another choice?

### Night 1 vs. night 2

People often sleep worse in the first night with the equipment (the **first-night effect**). Is that the case here?

👉 Change `METRIC` and rerun.

In [ ]:
METRIC = "SE"  # 👉 change me

paired = cohort.pivot_table(index="subject", columns="night", values=METRIC).dropna()
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot([1, 2], paired[[1, 2]].T, color="gray", alpha=0.4, lw=0.8)
ax.plot([1, 2], paired[[1, 2]].median(), color="black", marker="o", lw=2, label="median")
ax.set(xticks=[1, 2], xticklabels=["night 1", "night 2"], ylabel=METRIC, title=f"{len(paired)} subjects")
ax.legend()
plt.show()
print(f"Night 2 - night 1: median {(paired[2] - paired[1]).median():.1f}")

## 5. Band power and spindles/slow waves vs. age

Same plots as above, with the new features. Spearman's ρ with age; each dot is a night (remember the two-nights problem).

👉 Change `METRICS_NEW` and rerun: any column of the table, e.g. `delta_rel_N3_Fpz`, `sigma_abs_N2_Fpz`, `alpha_rel_WAKE_Pz`, `sp_density_N2_Fpz`, `sw_density_N3_Fpz`, `sw_coupling_strength_Fpz`, `yasa_sleep_intrusion`, `yasa_KL_mean_N3`.

In [ ]:
METRICS_NEW = ["delta_rel_N3_Fpz", "theta_rel_REM_Fpz", "alpha_rel_WAKE_Pz", "sigma_rel_N2_Fpz", "beta_rel_N2_Fpz",
               "sp_density_N2_Fpz", "sp_amplitude_N2_Fpz", "sw_density_N3_Fpz", "sw_ptp_Fpz",
               "yasa_kappa", "yasa_H_mean_N2", "yasa_KL_mean_N2", "yasa_KL_mean_WAKE", "yasa_sleep_intrusion", "yasa_wake_intrusion"]

fig, axes = plt.subplots(3, 5, figsize=(16, 9), sharex=True)
for ax, m in zip(axes.flat, METRICS_NEW):
    ax.scatter(cohort["age"], cohort[m], s=8, alpha=0.6)
    rho, _ = spearmanr(cohort["age"], cohort[m], nan_policy="omit")
    ax.set_title(f"{m}\n(ρ = {rho:.2f}, n = {cohort[m].notna().sum()})", fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel("Age (years)")
plt.tight_layout()
plt.show()

**Questions**
1. Which of the new features change most with age? Do spindle density and N3 slow-wave density fall with age, as in the literature (T8)?
2. Spindle density is zero or `NaN` in some old people. What does that do to the correlation, and to `sp_amplitude_N2_Fpz`?
3. A night with many bad epochs has fewer clean epochs (`n_clean_N2`, `clean_min_N2`). How does that affect the median power? And the density? Plot the feature against `n_clean_N2` to see.
4. YASA's agreement with the experts (`yasa_kappa`) changes with age. Are `yasa_sleep_intrusion` and `yasa_H_mean_N2` then a property of the brain or of the model?

## What's next

This table feeds Wednesday's age prediction and Thursday's dashboard.